# Generate MongoDB query load

Run concurrent queries through warm-up, measurement and drain phases. Inspect the results and collect counts through a callback.

Start Docker. Use the BONDI environment from this checkout, which includes `generate_load()`. Run the cells in order.
This demo creates `notebook-load-generation` on port 27125 and loads eight fictional businesses.
Change the name or port if another deployment uses them.

The final cell removes this demo deployment and its data. If you interrupt the notebook, run the cleanup cell manually.
The small dataset demonstrates the API. It does not measure realistic database performance.

See the [load generation guide](../docs/load-generation.md) and [API reference](../docs/reference/load.md).

In [ ]:
from collections import Counter
from statistics import mean

from bondi.deployment import deploy_mongodb
from bondi.load import Sequential, UniformRandom, WeightedRandom
from bondi.mongodb import MongoCollectionRef, MongoExecutor, MongoFindQuery
from bondi.workload import Workload, WorkloadEntry

DEPLOYMENT_NAME = "notebook-load-generation"
PORT = 27125
CLIENTS = 4
WARMUP_SECONDS = 1
MEASUREMENT_SECONDS = 3
DRAIN_SECONDS = 8
QUERY_TIMEOUT_SECONDS = 5

deployment = deploy_mongodb(name=DEPLOYMENT_NAME, port=PORT)
deployment.load_demo_data()

## Build a workload and choose a policy

Each client selects a query after its previous query and callback finish.
The weights below select Sydney twice as often as each other query, on average.
Actual counts depend on random selection and query runtime.

Replace `selection` with `UniformRandom(seed=42)` for equal weights, or `Sequential()` to repeat queries in order.
Each client has its own random stream or sequential position. Every sequential client starts with the first query.

In [ ]:
business = MongoCollectionRef("demo", "business")
labels = ["Sydney", "Melbourne", "Brisbane"]
workload = Workload([WorkloadEntry(MongoFindQuery(business, {"city": city})) for city in labels])
policies = {
    "weighted": WeightedRandom(weights=[2, 1, 1], seed=42),
    "uniform": UniformRandom(seed=42),
    "sequential": Sequential(),
}
selection = policies["weighted"]

## Generate load

Warm-up and measurement share one clock. Queries can cross phase boundaries.
Drain stops new queries and gives existing queries time to finish.
The drain allowance includes time for the health check, query timeout and cleanup.
Cancellation is cooperative, so drain expiry does not guarantee an immediate return.

The callback receives each `LoadResult`, including warm-up, errors and unfinished attempts at drain expiry.
This callback counts outcomes without printing every query. Slow callbacks can reduce the query rate.
`collect_results=False` reads all rows without keeping documents. `return_load_results=True` keeps the observations for inspection.

In [ ]:
outcome_counts = Counter()


def count_result(observation):
    outcome = (
        observation.result.status if observation.result is not None else observation.stop_reason
    )
    outcome_counts[(observation.phase_started, outcome)] += 1


with MongoExecutor.from_uri(
    deployment.uri,
    concurrency=CLIENTS,
    timeout_seconds=QUERY_TIMEOUT_SECONDS,
    health_check_timeout_seconds=2,
    continue_on_error=True,
) as executor:
    results = executor.generate_load(
        workload,
        selection=selection,
        warmup_seconds=WARMUP_SECONDS,
        measurement_seconds=MEASUREMENT_SECONDS,
        drain_timeout_seconds=DRAIN_SECONDS,
        collect_results=False,
        return_load_results=True,
        on_result=count_result,
    )

for (phase, outcome), count in sorted(outcome_counts.items()):
    print(f"{phase}: {outcome} = {count:,}")
assert sum(outcome_counts.values()) == len(results)

## Inspect measurement results

Select attempts that started during measurement, including those that finished during drain.
The table shows successful query runtimes from `ExecutionResult.elapsed_ms`. It excludes errors and unfinished attempts.
`LoadResult.duration` also includes the health check and scheduling after admission. It excludes callback time.
These are different measurements, so choose the one your experiment needs.

In [ ]:
measurement = [r for r in results if r.phase_started == "measurement"]
print(f"Measurement attempts: {len(measurement):,}")
print(f"{'Query':<12} {'Attempts':>10} {'Successes':>10} {'Mean query ms':>15}")
for position, label in enumerate(labels):
    attempts = [r for r in measurement if r.workload_position == position]
    successful = [r for r in attempts if r.result is not None and r.result.status == "success"]
    runtime = f"{mean(r.result.elapsed_ms for r in successful):.3f}" if successful else "n/a"
    print(f"{label:<12} {len(attempts):>10,} {len(successful):>10,} {runtime:>15}")

if measurement:
    example = measurement[0]
    print(f"Example attempt duration: {example.duration:.6f} seconds")
    print(f"Stop reason: {example.stop_reason}")

## Count results without retaining the list

For longer runs, use `return_load_results=False`. Callbacks still receive every observation, but the method returns an empty list.
This example keeps only outcome counts and uses sequential selection.

In [ ]:
outcome_counts.clear()
with MongoExecutor.from_uri(
    deployment.uri,
    concurrency=CLIENTS,
    timeout_seconds=QUERY_TIMEOUT_SECONDS,
    health_check_timeout_seconds=2,
    continue_on_error=True,
) as executor:
    streamed_results = executor.generate_load(
        workload,
        selection=Sequential(),
        measurement_seconds=1,
        drain_timeout_seconds=DRAIN_SECONDS,
        return_load_results=False,
        on_result=count_result,
    )

assert streamed_results == []
print(dict(outcome_counts))

## Select one pattern three times as often

Pattern 10 contains three city queries. Pattern 20 contains one rating query.
Give pattern 20 three times the weight of pattern 10, regardless of the number of instances.
The expected shares are 25% for pattern 10 and 75% for pattern 20.
`ByPattern` first selects the pattern, then selects an instance uniformly within that pattern.

The final table compares expected and observed shares. It counts all attempts, including failures and drain expiry.
The observed ratio should be near 3:1, not exactly 3:1. Random selection and the run duration affect the counts.


In [ ]:
from bondi.load import ByPattern

pattern_workload = Workload(
    [WorkloadEntry(entry.query, pattern_id=10) for entry in workload]
    + [WorkloadEntry(MongoFindQuery(business, {"stars": {"$gte": 4}}), pattern_id=20)]
)
pattern_counts = Counter()


def count_pattern(observation):
    pattern_id = pattern_workload[observation.workload_position].pattern_id
    pattern_counts[pattern_id] += 1


with MongoExecutor.from_uri(
    deployment.uri,
    concurrency=CLIENTS,
    timeout_seconds=QUERY_TIMEOUT_SECONDS,
    health_check_timeout_seconds=2,
    continue_on_error=True,
) as executor:
    executor.generate_load(
        pattern_workload,
        selection=ByPattern(
            pattern_ids=[10, 20],
            patterns=WeightedRandom(weights=[1, 3], seed=42),
            instances=UniformRandom(seed=43),
        ),
        measurement_seconds=2,
        drain_timeout_seconds=DRAIN_SECONDS,
        return_load_results=False,
        on_result=count_pattern,
    )

expected_shares = {10: 0.25, 20: 0.75}
total = pattern_counts.total()
print(f"{'Pattern':<10} {'Attempts':>10} {'Expected':>12} {'Observed':>12}")
for pattern_id, expected in expected_shares.items():
    count = pattern_counts[pattern_id]
    observed = f"{count / total:.1%}" if total else "n/a"
    print(f"{pattern_id:<10} {count:>10,} {expected:>12.1%} {observed:>12}")

print("Expected pattern 20 / pattern 10 ratio: 3.00:1")
if pattern_counts[10]:
    ratio = pattern_counts[20] / pattern_counts[10]
    print(f"Observed pattern 20 / pattern 10 ratio: {ratio:.2f}:1")
else:
    print("Observed ratio: n/a (no pattern 10 attempts)")

## Remove the demo deployment

This removes the named demo deployment, its data volume and its saved configuration.

In [ ]:
deployment.remove()